# Notebook 1 — Preprocessing and EDA

Takes the label files notebook 0 wrote into `own_footprints/` and turns them
into model-ready image patches. It:

1. **Downloads Sentinel imagery once** per city and assessment date, straight
   to Drive. If a raster is already there the export is skipped, so Earth
   Engine is only needed the first time.
2. **Cuts one patch per building** and saves the arrays plus the building
   geometries to `processed/`, so notebooks 2 and 3 need no Earth Engine.
3. **Explores the data** — damage over time, class balance, and what the
   radar signal actually looks like.

`CITY_REGISTRY` in `pipeline.py` currently holds **Gaza only**, with its nine
assessment dates. Get one city working end to end, then uncomment the others.

All settings describing the files on disk live in `PREP` in `pipeline.py`.
Changing patch size, sensors or imagery windows means editing them there and
re-running this notebook — file names encode the settings, so nothing is
silently overwritten.

## 1. Setup

Two things to check in `pipeline.py` before running:

```python
DATA_DIR = os.path.join(BASE, "Data", "own_footprints")   # not one_month
PREP["imagery"]["post_direction"] = "backward"
```

**Why backward.** Gaza's assessments are 19 to 64 days apart, and two gaps are
shorter than a month. A *forward* one-month window starting at the 15 October
assessment runs to 15 November — past the 7 November assessment — so it would
contain damage the October label says has not happened yet, and the model
would be punished for seeing it. Looking backward from each assessment date
avoids that entirely.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q geedim geemap

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
BASE = "/content/drive/MyDrive/War-Damage-Detection"   # adjust if needed
sys.path.append(BASE)

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

from pipeline import (PREP, CITY_REGISTRY, DATA_DIR, PWTT_DIR, PROCESSED_DIR,
                      S1_CHANNELS, S2_CHANNELS,
                      load_labels, list_label_dates, sample_buildings,
                      raster_path, processed_paths, save_processed,
                      load_processed, summarize_processed, shared_buildings)

print("label folder: ", DATA_DIR)
print("patch folder: ", PROCESSED_DIR)
print("\npreprocessing settings:")
for k, v in PREP.items():
    print(f"  {k}: {v}")

print("\nregistered cities and dates:")
for city, info in CITY_REGISTRY.items():
    on_disk = list_label_dates(city)
    missing = [d for d in info["label_dates"] if d not in on_disk]
    extra = [d for d in on_disk if d not in info["label_dates"]]
    print(f"  {city}: {len(info['label_dates'])} dates, "
          f"war start {info['war_start']}")
    print(f"     {info['label_dates']}")
    if missing:
        print(f"     MISSING from {DATA_DIR}: {missing}")
    if extra:
        print(f"     on disk but not registered: {extra}")

## 2. Earth Engine export — runs once per city and date

**Sentinel-1 (radar)** gives a 4-band stack: `pre_VV, pre_VH, post_VV,
post_VH` in decibels. Pre and post come from the **same relative orbit**,
because mixing orbits injects viewing-angle differences that look like change
but are not damage. The orbit is chosen by *measured coverage of the city*,
not by image count, since a scene clipping one corner counts the same as one
covering it fully.

**Sentinel-2 (optical)** gives an 8-band stack: pre and post B2, B3, B4, B8 at
10 m. Surface reflectance only exists from about March 2017, so older cities
fall back to top-of-atmosphere and the exporter says so.

The **pre-war window is identical for every date** (the 12 months before
`war_start`); only the post window moves. Each date therefore compares the
same pre-war baseline against a later snapshot — a cumulative label matched to
a cumulative signal.

Exports land directly on Drive, and re-running costs nothing once the files
exist. Budget roughly **250 MB per date** for Gaza, so about 2.2 GB for nine.

In [5]:
S1_EARLIEST = "2014-10-01"   # Sentinel-1 IW data barely exists before this


def _init_ee():
    import ee
    project = PREP["imagery"]["gee_project"]
    try:
        ee.Initialize(project=project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=project)
    return ee


def _iso(date):
    """'20240503' -> '2024-05-03'."""
    return f"{date[:4]}-{date[4:6]}-{date[6:]}"


def _windows(ee, war_start, label_date):
    """Pre-war and post-assessment date windows as Earth Engine dates."""
    im = PREP["imagery"]
    war, label = ee.Date(war_start), ee.Date(label_date)
    pre = (war.advance(-im["pre_months"], "month"), war)
    if im["post_direction"] == "forward":
        post = (label, label.advance(im["post_months"], "month"))
    else:
        post = (label.advance(-im["post_months"], "month"), label)
    return pre, post


def city_bbox(city, date):
    """Export area: bounding box of the labelled buildings plus a buffer."""
    gdf = load_labels(city, date)
    minx, miny, maxx, maxy = gdf.total_bounds
    b = PREP["imagery"]["aoi_buffer_deg"]
    return [minx - b, miny - b, maxx + b, maxy + b]

In [6]:
def export_s1(city, date, bbox):
    """Export the 4-band radar stack for one city and date (skips if on Drive)."""
    out = raster_path(city, date, "s1")
    if os.path.exists(out):
        print(f"  {os.path.basename(out)} exists, skipping export")
        return out

    ee = _init_ee()
    import geemap
    im = PREP["imagery"]
    aoi = ee.Geometry.Rectangle(bbox)
    (pre_a, pre_b), (post_a, post_b) = _windows(
        ee, CITY_REGISTRY[city]["war_start"], _iso(date))

    if pre_a.format("YYYY-MM-dd").getInfo() < S1_EARLIEST:
        print(f"  WARNING: the pre window starts before {S1_EARLIEST}, where "
              f"Sentinel-1 has no data. Shorten pre_months or move war_start.")

    # Pixels outside the radar swath are minus infinity in dB (log of zero).
    # Mask them before compositing, or one pixel poisons the median.
    def mask_edges(img):
        valid = img.select("VV").gt(-35).And(img.select("VH").gt(-35))
        return img.updateMask(valid)

    s1 = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi)
          .filter(ee.Filter.eq("instrumentMode", "IW"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
          .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
          .select(["VV", "VH"]).map(mask_edges))

    def orbit_counts(a, b):
        h = (s1.filterDate(a, b)
               .aggregate_histogram("relativeOrbitNumber_start").getInfo())
        return {int(float(k)): int(v) for k, v in h.items()}

    pre_c, post_c = orbit_counts(pre_a, pre_b), orbit_counts(post_a, post_b)
    shared = set(pre_c) & set(post_c)
    if not shared:
        raise RuntimeError(f"{city} {date}: no orbit covers both windows. "
                           f"pre={pre_c} post={post_c}")

    # Image count is not coverage: measure what fraction of the city each
    # orbit actually observes, and take the best worst-case.
    def coverage(a, b, orbit):
        c = s1.filterDate(a, b).filter(
            ee.Filter.eq("relativeOrbitNumber_start", orbit))
        m = c.select("VV").count().gt(0).unmask(0)
        return m.reduceRegion(ee.Reducer.mean(), aoi, 200, maxPixels=1e9).get("VV")

    cov = ee.Dictionary({
        f"{o}_{w}": coverage(a, b, o)
        for o in shared
        for w, (a, b) in [("pre", (pre_a, pre_b)), ("post", (post_a, post_b))]
    }).getInfo()

    scored = {o: min(cov[f"{o}_pre"] or 0, cov[f"{o}_post"] or 0) for o in shared}
    best = max(scored, key=lambda o: (scored[o], post_c[o]))
    for o in sorted(shared):
        print(f"  orbit {o}: {pre_c[o]:3d} pre / {post_c[o]:3d} post images, "
              f"coverage {cov[f'{o}_pre'] or 0:.2f} / {cov[f'{o}_post'] or 0:.2f}")
    print(f"  using orbit {best}, min coverage {scored[best]:.2f}")
    if scored[best] < im["min_aoi_coverage"]:
        print(f"  WARNING: the best orbit covers only {scored[best]*100:.0f} "
              f"percent of the city, much of the export will be nodata.")
    if post_c[best] < 3:
        print(f"  WARNING: only {post_c[best]} post images, the median barely "
              f"suppresses radar speckle. Consider post_months = 2.")

    s1 = s1.filter(ee.Filter.eq("relativeOrbitNumber_start", best))
    pre = s1.filterDate(pre_a, pre_b).median().rename(["pre_VV", "pre_VH"])
    post = s1.filterDate(post_a, post_b).median().rename(["post_VV", "post_VH"])
    stack = pre.addBands(post).clip(aoi).toFloat()
    geemap.download_ee_image(stack, out, region=aoi, scale=im["scale"],
                             crs="EPSG:4326")
    print(f"  exported {os.path.basename(out)}")
    return out

In [7]:
def export_s2(city, date, bbox):
    """Export the 8-band optical stack for one city and date (skips if on Drive)."""
    out = raster_path(city, date, "s2")
    if os.path.exists(out):
        print(f"  {os.path.basename(out)} exists, skipping export")
        return out

    ee = _init_ee()
    import geemap
    im = PREP["imagery"]
    aoi = ee.Geometry.Rectangle(bbox)
    bands = ["B2", "B3", "B4", "B8"]
    (pre_a, pre_b), (post_a, post_b) = _windows(
        ee, CITY_REGISTRY[city]["war_start"], _iso(date))

    def mask_scl(img):        # surface reflectance cloud mask
        scl = img.select("SCL")
        bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10))
        return img.updateMask(bad.Not())

    def mask_qa60(img):       # top-of-atmosphere cloud mask
        qa = img.select("QA60")
        clear = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
        return img.updateMask(clear)

    def composite(a, b):
        for cid, fn, level in [("COPERNICUS/S2_SR_HARMONIZED", mask_scl, "SR"),
                               ("COPERNICUS/S2_HARMONIZED", mask_qa60, "TOA")]:
            col = (ee.ImageCollection(cid).filterBounds(aoi).filterDate(a, b)
                   .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",
                                        im["s2_max_cloud"])))
            if col.size().getInfo() > 0:
                return col.map(fn).select(bands).median(), col.size().getInfo(), level
        return None, 0, "none"

    pre, n_pre, lvl_pre = composite(pre_a, pre_b)
    post, n_post, lvl_post = composite(post_a, post_b)
    print(f"  Sentinel-2 scenes: {n_pre} pre ({lvl_pre}) / {n_post} post ({lvl_post})")
    if n_pre == 0 or n_post == 0:
        raise RuntimeError(f"{city} {date}: no usable Sentinel-2 scenes in one "
                           f"window. Raise s2_max_cloud or widen the windows.")
    if lvl_pre != lvl_post:
        print("  WARNING: pre and post use different processing levels (SR vs "
              "TOA), so their difference is contaminated by the atmosphere.")

    stack = (pre.rename([f"pre_{b}" for b in bands])
                .addBands(post.rename([f"post_{b}" for b in bands]))
                .clip(aoi).toFloat())
    geemap.download_ee_image(stack, out, region=aoi, scale=im["scale"],
                             crs="EPSG:4326")
    print(f"  exported {os.path.basename(out)}")
    return out


def ensure_rasters(city, date):
    """Return {'s1': path or None, 's2': path or None}, exporting what is missing."""
    exporters = {"s1": export_s1, "s2": export_s2}
    paths = {"s1": None, "s2": None}
    missing = [s for s in PREP["sensors"]
               if not os.path.exists(raster_path(city, date, s))]
    bbox = city_bbox(city, date) if missing else None
    for sensor in PREP["sensors"]:
        paths[sensor] = (exporters[sensor](city, date, bbox) if sensor in missing
                         else raster_path(city, date, sensor))
    return paths

## 3. Cut patches and save to Drive

One patch per building, centred on its centroid. Buildings whose patch falls
off the raster edge or touches a non-finite pixel are dropped, so every model
later sees the same clean set. (Radar nodata is *minus infinity* in dB, not
NaN — hence the `np.isfinite` check.)

**The subsample is chosen by building id, not row position.** With nine dates
this matters: notebook 2 matches buildings across dates on `system:index`, so
every date must contain the same buildings. `sample_buildings` picks ids from
a sorted list with a fixed seed, which is identical across dates by
construction rather than by luck.

Saved per city and date:

* `processed/….npz` — patches `X`, labels `y`, coordinates (`lat`, `lon`, and
  `xy` in metres for the spatial smoothing), channel names
* `processed/….parquet` — the geometries with label, severity, confidence and
  `system:index`

In [ ]:
def preprocess_city(city, date):
    """Rasters -> per-building patches, saved to Drive. Skips existing output."""
    _, _, x_path = processed_paths(city, date)
    if os.path.exists(x_path):
        print(f"{city} {date}: already processed")
        return

    patch = PREP["patch_size"]
    half = patch // 2
    gdf = load_labels(city, date)
    gdf = sample_buildings(gdf, PREP["n_sample"], PREP["seed"]).copy()

    paths = ensure_rasters(city, date)
    rasters, names = [], []
    for sensor, chans in [("s1", S1_CHANNELS), ("s2", S2_CHANNELS)]:
        if paths[sensor]:
            with rasterio.open(paths[sensor]) as src:
                rasters.append((src.read().astype(np.float32),
                                ~src.transform, src.height, src.width))
            names += chans

    lons = gdf["centroid"].apply(lambda p: p.x).values
    lats = gdf["centroid"].apply(lambda p: p.y).values

    # top-left corner of each building's patch in every raster
    corner_list = []
    for arr, inv, H, W in rasters:
        cols, rows = inv * (lons, lats)
        r0 = np.round(rows).astype(int) - half
        c0 = np.round(cols).astype(int) - half
        inside = (r0 >= 0) & (c0 >= 0) & (r0 + patch <= H) & (c0 + patch <= W)
        corner_list.append((r0, c0, inside))
    ok = np.logical_and.reduce([c[2] for c in corner_list])

    X, keep = [], []
    idx = gdf.index.values
    for j in np.where(ok)[0]:
        parts, good = [], True
        for (arr, inv, H, W), (r0, c0, _) in zip(rasters, corner_list):
            p = arr[:, r0[j]:r0[j] + patch, c0[j]:c0[j] + patch]
            if not np.isfinite(p).all():
                good = False
                break
            parts.append(p)
        if good:
            X.append(np.concatenate(parts, axis=0))
            keep.append(idx[j])

    if not X:
        raise RuntimeError(f"{city} {date}: no usable patches. The raster is "
                           f"probably mostly nodata, check the coverage warning "
                           f"printed by the export above.")

    sub = gdf.loc[np.array(keep)]
    merc = sub["centroid"].to_crs(3857)          # metres, for spatial smoothing
    d = {
        "city": city, "date": date,
        "X": np.stack(X),
        "y": sub["class"].to_numpy(int),
        "lat": sub["centroid"].apply(lambda p: p.y).values,
        "lon": sub["centroid"].apply(lambda p: p.x).values,
        "xy": np.c_[merc.apply(lambda p: p.x), merc.apply(lambda p: p.y)],
        "channel_names": names,
        "gdf": sub,
    }
    print(f"{city} {date}: kept {len(sub):,} of {len(gdf):,} buildings, "
          f"{d['y'].mean()*100:.1f} percent damaged, {len(names)} channels")
    save_processed(d)


for city, info in CITY_REGISTRY.items():
    for date in info["label_dates"]:
        preprocess_city(city, date)

In [ ]:
# The same buildings must survive at every date, or the temporal label
# matching in notebook 2 silently drops rows. A patch is kept only if it fits
# inside its raster and touches no nodata, and the rasters differ between
# dates, so check. This reads only the geometry files, never the pixels.
for city, info in CITY_REGISTRY.items():
    shared, sets = shared_buildings(city)
    print(f"{city}: {len(shared):,} buildings present at all {len(sets)} dates")
    for date, s in sets.items():
        extra = len(s) - len(shared)
        print(f"   {date}: {len(s):,}" + (f"   ({extra:,} not shared)" if extra else ""))
    if len(shared) < 0.95 * max(len(s) for s in sets.values()):
        print("   NOTE: a date is losing buildings to nodata. More scenes in "
              "the post composite fixes it - try post_months = 2.")

## 4. EDA

Everything below reads the processed files, so this section runs without an
Earth Engine session.

In [ ]:
CITY = "Gaza"
dates = CITY_REGISTRY[CITY]["label_dates"]

# summarize_processed reads only the label arrays, so this is instant even
# though the patch files are gigabytes. Only the newest date's patches are
# opened, and even then as a memory map rather than a copy in RAM.
summary = summarize_processed(CITY)
print(summary.to_string(index=False))

latest = load_processed(CITY, dates[-1])
print(f"\nchannels: {latest['channel_names']}")
print(f"patch array: {latest['X'].shape}, dtype {latest['X'].dtype}, "
      f"{latest['X'].nbytes / 1e9:.2f} GB on disk (memory mapped, not loaded)")

In [ ]:
# Damage over time. Left: the cumulative share. Right: buildings newly
# labelled damaged since the previous assessment - which is where the new
# information in each date actually sits.
x = pd.to_datetime(dates)
new = np.diff(summary["damaged"].to_numpy(), prepend=0)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(x, summary["damaged %"], marker="o", color="C3")
ax[0].set_ylabel("percent damaged")
ax[0].set_title(f"{CITY}: cumulative damage")
ax[1].bar(x, new, width=18, color="C0")
ax[1].set_ylabel("buildings")
ax[1].set_title("newly labelled damaged since previous assessment")
for a in ax:
    a.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Where the damage is, at each assessment date. It is clearly clustered,
# which is the argument for both the latitude-band splits (no leakage between
# train and test) and the spatial smoothing (use the clustering at prediction
# time). Only coordinates and labels are read here, never pixels.
fig, axes = plt.subplots(1, len(dates), figsize=(4 * len(dates), 6))
axes = np.atleast_1d(axes)
for ax, d in zip(axes, dates):
    npz_path, _, _ = processed_paths(CITY, d)
    with np.load(npz_path, allow_pickle=False) as z:
        lat, lon, y = z["lat"], z["lon"], z["y"]
    ax.scatter(lon[y == 0], lat[y == 0], s=1.5, alpha=0.35, color="C0",
               label="intact")
    ax.scatter(lon[y == 1], lat[y == 1], s=1.5, alpha=0.35, color="C3",
               label="damaged")
    ax.set_title(f"{d}  ({y.mean() * 100:.1f} percent)")
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(markerscale=6, loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
# The actual signal. A damaged building should look different AFTER the war
# relative to its OWN pre-war baseline, so the informative quantity is the
# post-minus-pre difference rather than either level alone. Decibels,
# averaged over each patch.
f, ch, y = latest, latest["channel_names"], latest["y"]

# np.asarray on one channel of the memory map reads just that
# channel, about 450 MB rather than the whole 1.8 GB file.
pre_vv = f["X"][:, ch.index("s1_pre_VV")].mean(axis=(1, 2))
post_vv = f["X"][:, ch.index("s1_post_VV")].mean(axis=(1, 2))
diff = post_vv - pre_vv

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, (v, name) in zip(ax, [(pre_vv, "pre-war VV"), (post_vv, "post VV"),
                             (diff, "post minus pre VV")]):
    a.hist(v[y == 0], bins=60, alpha=0.6, density=True, label="intact")
    a.hist(v[y == 1], bins=60, alpha=0.6, density=True, label="damaged")
    a.set_xlabel("dB")
    a.set_title(name)
ax[2].axvline(0, color="k", ls="--", lw=1)
ax[0].legend()
plt.suptitle(f"{CITY} {dates[-1]}: patch-mean backscatter", y=1.04)
plt.tight_layout()
plt.show()

print(f"mean post-minus-pre, intact:  {diff[y == 0].mean():+.2f} dB")
print(f"mean post-minus-pre, damaged: {diff[y == 1].mean():+.2f} dB")
print("The classes overlap heavily, which is exactly why we hand a CNN the "
      "whole patch instead of a single summary number.")

In [ ]:
# Example patches: post-event VV for damaged and intact buildings, on a
# shared colour scale so they are comparable.
rng = np.random.default_rng(0)
post_idx = ch.index("s1_post_VV")
dmg = rng.choice(np.where(y == 1)[0], 5, replace=False)
ok = rng.choice(np.where(y == 0)[0], 5, replace=False)
# percentiles from a sample, so the whole channel is not read
sample = np.asarray(f["X"][:5000, post_idx])
vmin, vmax = np.percentile(sample, [2, 98])

fig, axes = plt.subplots(2, 5, figsize=(12, 5.2))
for row, (sel, label) in enumerate([(dmg, "damaged"), (ok, "intact")]):
    for col, j in enumerate(sel):
        axes[row, col].imshow(np.asarray(f["X"][j, post_idx]), cmap="gray",
                              vmin=vmin, vmax=vmax)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
    axes[row, 0].set_ylabel(label)
fig.suptitle(f"{CITY} {dates[-1]}: post-event VV, "
             f"{PREP['patch_size']} px = {PREP['patch_size'] * 10} m across")
plt.tight_layout()
plt.show()

In [ ]:
# Label quality, using the extra columns notebook 0 carried through.
g = latest["gdf"]

if "severity" in g.columns and g["severity"].notna().any():
    print("UNOSAT severity of damaged buildings (1 = most severe):")
    print(g["severity"].value_counts().sort_index().to_string())

if "damage_pts" in g.columns:
    d1 = g.loc[g["class"] == 1, "damage_pts"]
    print(f"\ndamage points per damaged building: median {d1.median():.0f}, "
          f"max {int(d1.max())}")

# Do bigger buildings get labelled damaged more often? If strongly so, size
# alone carries label information, and the model could lean on patch texture
# that reflects size rather than damage.
by_size = (g.groupby(pd.qcut(g["area"], 5, duplicates="drop"), observed=True)
             ["class"].agg(["mean", "size"]))
by_size["mean"] = (by_size["mean"] * 100).round(1)
print("\ndamaged percent by building-size quintile:")
print(by_size.rename(columns={"mean": "damaged %", "size": "n"}).to_string())

In [ ]:
# The PWTT baseline. Our own label files have max_change empty, so this reads
# the published benchmark file directly - enough to see how separable the
# statistic is on its own. The dashed line is its published threshold of 3.3.
bench = sorted(glob.glob(os.path.join(PWTT_DIR, f"{CITY}_*_footprints.csv")))
if not bench:
    print(f"no benchmark file for {CITY} in {PWTT_DIR}, skipping")
else:
    b = pd.read_csv(bench[-1], usecols=["class", "max_change"], low_memory=False)
    print(f"{os.path.basename(bench[-1])}: {len(b):,} buildings, "
          f"{b['class'].mean() * 100:.2f} percent damaged")
    plt.figure(figsize=(6, 3.4))
    plt.hist(b.loc[b["class"] == 0, "max_change"], bins=60, alpha=0.6,
             density=True, label="intact")
    plt.hist(b.loc[b["class"] == 1, "max_change"], bins=60, alpha=0.6,
             density=True, label="damaged")
    plt.axvline(3.3, color="k", ls="--", lw=1)
    plt.xlabel("PWTT max_change (T statistic)")
    plt.title("the baseline we are trying to beat")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Done

On Drive now:

* `rasters/` — the Sentinel exports, one per city and date
* `processed/` — one `.npz` + `.parquet` per city and date

Continue with **notebook 2**. With nine Gaza dates registered you can set
`label_temporal: True` there, which trains on every date with labels
propagated forward in time. Keep validation and test on a **single** date so
the numbers stay comparable — dates are not independent samples, and holding
out a later date from an earlier one would overlap almost completely.

Once Gaza runs end to end, uncomment the other cities in `CITY_REGISTRY` and
re-run this notebook: only the new exports and patches get computed.